In [1]:
import os
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.nvidia import NVIDIAEmbedding
import chromadb
from dotenv import load_dotenv
from functions.rule_downloader import download_rules

# Load environment variables from .env file (need parentheses to actually call the function)
load_dotenv()
NVIDIA_API_KEY = os.environ["NVIDIA_API_KEY"]

In [2]:
# Download updated rules from MTG website and save them in the "documents" directory 
if download_rules():
    print("Regole scaricate con successo.") 

# Load all documents from the "documents" directory
# This reads PDFs, text files, etc. and converts them into Document objects
documents = SimpleDirectoryReader(input_dir="documents").load_data()

URL found: https://media.wizards.com/2026/downloads/MagicCompRules%2020260417.txt
Downloaded: MagicCompRules_20260417.txt
Regole scaricate con successo.


In [3]:
# Create a persistent Chroma client that saves data to disk at "./chroma_db"
# This ensures the database persists between script runs
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [4]:
# Get existing collection or create new one named "documents_collection"
# Collections in Chroma are like tables in a database
chroma_collection = chroma_client.get_or_create_collection("documents_collection")

In [5]:
# Wrap the Chroma collection in LlamaIndex's ChromaVectorStore adapter
# This allows LlamaIndex to interact with Chroma's storage
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [6]:
# Create a StorageContext that tells LlamaIndex where to store vectors
# This is the critical piece that connects the index to persistent storage
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [7]:
# Initialize NVIDIA embedding model to convert text into vectors
# nvidia/nv-embed-v1 is cost-effective and works well for most use cases
embed_model = NVIDIAEmbedding(model="nvidia/nv-embed-v1", api_key=NVIDIA_API_KEY)

In [8]:
# Create the index from documents and store vectors in Chroma
# The storage_context ensures embeddings are saved to the persistent database
# Without storage_context, embeddings would only exist in memory
index = VectorStoreIndex.from_documents(
    documents, 
    storage_context=storage_context,
    embed_model=embed_model
)

2026-06-02 16:15:53,640 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-02 16:15:55,499 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-02 16:16:01,558 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-02 16:16:04,838 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-02 16:16:06,341 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-02 16:16:07,905 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-02 16:16:09,453 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-02 16:16:29,508 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-02 16:16:30,962 - INFO - HTTP Request: POST https://inte